# Rede neural convolucional com transformada Wavelet no pipeline 


## 1. Dataset - MiniMIAS

### 1.1 Rodar localmente o programa

- Para isso é necessário baixar e extrair o dataset para a máquina local. 

**Link:** https://www.repository.cam.ac.uk/items/b6a97f0c-3b9b-40ad-8f18-3d121eef1459

- Baixe o dataset em Files;
- Após extrair em uma pasta os arquivos. Comando via terminal: _unzip MIAS-DB.zip_

In [5]:
import os

pasta = "dataset-mias/"

arquivos = os.listdir(pasta)
contador = 0

for arquivo in arquivos:
    if ".pgm" in arquivo:
        contador += 1

print(contador)

322


### 1.2 Classificando as imagnes

In [15]:
import os
import random
import shutil
from PyPDF2 import PdfReader

pasta_imagens = "dataset-mias/"

com_nodulo = []
sem_nodulo = []

# LER PDF
reader = PdfReader("dataset-mias/00README.pdf")
linhas = []

for pagina in reader.pages:
    texto = pagina.extract_text()
    linhas.extend(texto.split("\n"))

# PROCESSAR
for linha in linhas:
    if "mdb" in linha:
        partes = linha.split()
        nome = partes[0] + ".pgm"
        
        if "NORM" in linha:
            sem_nodulo.append(nome)
        else:
            com_nodulo.append(nome)

# DIVIDIR
def dividir(lista, pasta_train, pasta_test):
    random.shuffle(lista)
    corte = int(0.7 * len(lista))
    
    for nome in lista[:corte]:
        origem = os.path.join(pasta_imagens, nome)
        destino = os.path.join(pasta_train, nome)
        if os.path.exists(origem):
            shutil.copy(origem, destino)
    
    for nome in lista[corte:]:
        origem = os.path.join(pasta_imagens, nome)
        destino = os.path.join(pasta_test, nome)
        if os.path.exists(origem):
            shutil.copy(origem, destino)

# PASTAS
os.makedirs("dataset/train/com_nodulo", exist_ok=True)
os.makedirs("dataset/train/sem_nodulo", exist_ok=True)
os.makedirs("dataset/test/com_nodulo", exist_ok=True)
os.makedirs("dataset/test/sem_nodulo", exist_ok=True)

# EXECUTAR
dividir(com_nodulo, "dataset/train/com_nodulo", "dataset/test/com_nodulo")
dividir(sem_nodulo, "dataset/train/sem_nodulo", "dataset/test/sem_nodulo")

print("Finalizado!")

Finalizado!


### 1.3 Carregando dataset para CNN

In [18]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),  
    transforms.Resize((28, 28)), 
    transforms.ToTensor(),
])

train_dataset = datasets.ImageFolder(
    root="dataset/train",
    transform=transform
)

test_dataset = datasets.ImageFolder(
    root="dataset/test",
    transform=transform
)

# Criar DataLoader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

# Verificar Classes
print(train_dataset.classes)

ModuleNotFoundError: No module named 'torchvision'

## 2. Criando uma camada Wavelet usando PyTorch

In [16]:
# Camada Wavelet 
import torch
import torch.nn as nn
import torch.nn.functional as F

class WaveletLayer(nn.Module):
    def __init__(self):
        super(WaveletLayer, self).__init__()

        # Filtro Haar LL
        kernel = torch.tensor([[0.5, 0.5],
                               [0.5, 0.5]], dtype=torch.float32)

        # Ajustar formato: (out_channels, in_channels, H, W)
        kernel = kernel.view(1, 1, 2, 2)

        # Registrar como peso fixo (não treinável)
        self.register_buffer("weight", kernel)
        # para tornar a wavelet treinável pela própria cnn: self.weight = nn.Parameter(kernel)

    def forward(self, x):
        return F.conv2d(x, self.weight, stride=2)


ModuleNotFoundError: No module named 'torch'

## 3. Modelo CNN integrando a camada Wavelet

In [ ]:
class WaveletCNN(nn.Module):
    def __init__(self):
        super(WaveletCNN, self).__init__()

        # camada wavelet
        self.wavelet = WaveletLayer()
        # aprende padrões
        self.conv = nn.Conv2d(1, 16, 3, padding=1)
        self.fc = nn.Linear(16 * 14 * 14, 2) # o 2 representa o número de classes

    def forward(self, x):
        x = self.wavelet(x)  # Wavelet 
        x = torch.relu(self.conv(x)) # aplica não linearidade
        x = x.view(x.size(0), -1)
        x = self.fc(x) # toma decisão
        return x

### 3.1 Treinando o modelo

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

model = WaveletCNN()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(10):
    model.train()
    
    for imagens, labels in train_loader:
        outputs = model(imagens)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch} - Loss: {loss.item()}")

### 3.2 Testando o modelo

In [ ]:
model.eval()
corretos = 0
total = 0

with torch.no_grad():
    for imagens, labels in test_loader:
        outputs = model(imagens)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        corretos += (predicted == labels).sum().item()

print(f"Acurácia: {100 * corretos / total:.2f}%")